# **Machine Learning Pipeline: Heart Failure Prediction**

## **Setup & Import Library**

In [1]:
import os
import sys
import tensorflow as tf
import tensorflow_model_analysis as tfma
from tfx.components import (
    CsvExampleGen,
    StatisticsGen,
    SchemaGen,
    ExampleValidator,
    Transform,
    Tuner,
    Trainer,
    Evaluator,
    Pusher,
)
from tfx.proto import trainer_pb2, example_gen_pb2, pusher_pb2
from tfx.orchestration.experimental.interactive.interactive_context import (
    InteractiveContext,
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

# Menambahkan root folder path
sys.path.append("..")

2026-08-15 06:32:18.546937: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-15 06:32:27.239686: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-15 06:32:27.241010: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-15 06:32:38.051011: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## **Inisialisasi**

In [2]:
PIPELINE_NAME = "main-pipeline"
PIPELINE_ROOT = os.path.join("..", PIPELINE_NAME)

# Seluruh artifact TFX akan disimpan di folder
context = InteractiveContext(pipeline_name=PIPELINE_NAME, pipeline_root=PIPELINE_ROOT)

Tahap awal pembangunan machine learning pipeline dilakukan dengan menginisialisasi **`InteractiveContext`**. Komponen ini berfungsi untuk:

- **Orchestrator Interaktif:** Mengelola dan mengeksekusi setiap komponen TFX secara mandiri di dalam lingkungan notebook.
- **Pencatatan Metadata:** Melacak silsilah artefak dan mencatat riwayat eksekusi ke dalam basis data SQLite (`metadata.sqlite`) di direktori `main-pipeline`.

## **Data Ingestion**

In [3]:
DATA_ROOT = os.path.join("..", "data")

# Membagi dataset
output_config = example_gen_pb2.Output(
    split_config=example_gen_pb2.SplitConfig(
        splits=[
            example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
            example_gen_pb2.SplitConfig.Split(name="eval", hash_buckets=2),
        ]
    )
)

example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=output_config)
context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 1
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Komponen **`CsvExampleGen`** bertugas membaca dataset mentah dari direktori data dan mengonversinya ke dalam format biner standar TensorFlow (`tf.train.Example` berbasis TFRecord). Pada tahap ini, dataset diproses dengan alur berikut:

- **Injest Data:** Mengambil data CSV dari direktori `DATA_ROOT`.
- **Pembagian Dataset:** Membagi data secara deterministik menggunakan hash buckets menjadi dua bagian, yaitu **80% data latih** (`train`) dan **20% data evaluasi** (`eval`).
- **Konversi Format:** Mengubah struktur data menjadi TFRecord agar dapat dikonsumsi secara efisien oleh komponen TFX selanjutnya.

## **Data Validation**

In [4]:
# Menghitung statistik deskriptif dataset
statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])
context.run(statistics_gen)
context.show(statistics_gen.outputs["statistics"])

# Generate skema tipe data dan rentang nilai
schema_gen = SchemaGen(statistics=statistics_gen.outputs["statistics"])
context.run(schema_gen)
context.show(schema_gen.outputs["schema"])

# Memeriksa anomali data berdasarkan skema
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"], schema=schema_gen.outputs["schema"]
)
context.run(example_validator)
context.show(example_validator.outputs["anomalies"])

,Type,Presence,Valency,Domain
Feature name,,,,
'Age',INT,required,,-
'ChestPainType',STRING,required,,'ChestPainType'
'Cholesterol',INT,required,,-
'ExerciseAngina',STRING,required,,'ExerciseAngina'
'FastingBS',INT,required,,-
'HeartDisease',INT,required,,-
'MaxHR',INT,required,,-
'Oldpeak',FLOAT,required,,-
'RestingBP',INT,required,,-


,Values
Domain,
'ChestPainType',"'ASY', 'ATA', 'NAP', 'TA'"
'ExerciseAngina',"'N', 'Y'"
'RestingECG',"'LVH', 'Normal', 'ST'"
'ST_Slope',"'Down', 'Flat', 'Up'"
'Sex',"'F', 'M'"


Tahap validasi data bertujuan untuk menjamin kualitas data sebelum masuk ke proses pelatihan model melalui tiga komponen TFX utama:

- **`StatisticsGen`:** Menghitung statistik deskriptif (seperti nilai rata-rata, standar deviasi, nilai ekstrem, dan sebaran data kategorikal) untuk setiap kolom pada dataset train maupun eval.
- **`SchemaGen`:** Menganalisis hasil statistik untuk membuat skema data otomatis yang mendefinisikan tipe data, rentang nilai yang diizinkan, serta ekspektasi struktur fitur.
- **`ExampleValidator`:** Memeriksa keberadaan anomali, missing values, atau ketidaksesuaian format data dengan mencocokkan statistik data terhadap skema yang telah dibuat.

## **Feature Engineering**

In [5]:
TRANSFORM_MODULE_FILE = os.path.join("..", "modules", "transform.py")

transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=TRANSFORM_MODULE_FILE,
)
context.run(transform)

running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying trainer.py -> build/lib
copying tuner.py -> build/lib
copying transform.py -> build/lib
installing to /tmp/tmpb2y24r1i
running install
running install_lib
copying build/lib/tuner.py -> /tmp/tmpb2y24r1i
copying build/lib/transform.py -> /tmp/tmpb2y24r1i
copying build/lib/trainer.py -> /tmp/tmpb2y24r1i
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmpb2y24r1i/tfx_user_code_Transform-0.0+e9cb44594d6869e74ad

/workspaces/ml-pipeline/venv/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


Processing /workspaces/ml-pipeline/main-pipeline/_wheels/tfx_user_code_Transform-0.0+e9cb44594d6869e74ad0813cd4ed047f43032fa52601de7e6604f061eaecdca4-py3-none-any.whl



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Processing /workspaces/ml-pipeline/main-pipeline/_wheels/tfx_user_code_Transform-0.0+e9cb44594d6869e74ad0813cd4ed047f43032fa52601de7e6604f061eaecdca4-py3-none-any.whl


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 5
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Komponen **`Transform`** mengeksekusi logika prapemrosesan data yang terdefinisi secara terisolasi di dalam skrip `modules/transform.py`. Proses transformasi ini meliputi:

- **Feature Engineering:** Menerapkan standardisasi Z-score (`tft.scale_to_z_score`) pada fitur numerik dan pembuatan indeks kosakata (`tft.compute_and_apply_vocabulary`) pada fitur kategorikal.
- **Pencegahan Training-Serving Skew:** Membungkus seluruh logika transformasi ke dalam grafik prapemrosesan TensorFlow (preprocessing graph). Grafik ini akan ditempelkan pada model akhir sehingga prapemrosesan data saat training dan serving berjalan identik.

## **Hyperparameter Tuning**

In [6]:
TUNER_MODULE_FILE = os.path.join("..", "modules", "tuner.py")

tuner = Tuner(
    module_file=TUNER_MODULE_FILE,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=20),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=10),
)
context.run(tuner)

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.48750001192092896

Best val_accuracy So Far: 0.84375
Total elapsed time: 00h 00m 09s
INFO:tensorflow:Oracle triggered exit


INFO:tensorflow:Oracle triggered exit


Results summary
Results in ../main-pipeline/.temp/6/heart_failure_tuning
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
num_layers: 1
units_0: 96
dropout_0: 0.4
learning_rate: 0.01
Score: 0.84375

Trial 0 summary
Hyperparameters:
num_layers: 1
units_0: 128
dropout_0: 0.1
learning_rate: 0.01
Score: 0.8187500238418579

Trial 3 summary
Hyperparameters:
num_layers: 2
units_0: 96
dropout_0: 0.4
learning_rate: 0.001
units_1: 96
dropout_1: 0.30000000000000004
units_2: 32
dropout_2: 0.1
Score: 0.784375011920929

Trial 2 summary
Hyperparameters:
num_layers: 3
units_0: 32
dropout_0: 0.4
learning_rate: 0.001
units_1: 32
dropout_1: 0.1
units_2: 32
dropout_2: 0.1
Score: 0.659375011920929

Trial 4 summary
Hyperparameters:
num_layers: 3
units_0: 64
dropout_0: 0.2
learning_rate: 0.0001
units_1: 32
dropout_1: 0.30000000000000004
units_2: 32
dropout_2: 0.1
Score: 0.48750001192092896


ExecutionResult(
    component_id: Tuner
    execution_id: 6
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Komponen **`Tuner`** mengintegrasikan pustaka KerasTuner melalui skrip `modules/tuner.py` untuk mencari kombinasi hyperparameter optimal secara otomatis. Proses optimasi ini mencakup:

- **Penggunaan Data Terintegrasi:** Mengonsumsi data hasil transformasi (`transformed_examples`) beserta preprocessing graph (`transform_graph`) dari komponen `Transform`.
- **Eksplorasi Hyperparameter:** Menguji variasi arsitektur dan konfigurasi model, seperti jumlah dense layer, jumlah unit per layer, rasio dropout, serta learning rate.
- **Pencarian Kombinasi Terbaik:** Mengukur performa setiap eksperimen pada data evaluasi untuk memaksimalkan metrik `val_accuracy`.

## **Model Training**

In [7]:
TRAINER_MODULE_FILE = os.path.join("..", "modules", "trainer.py")

trainer = Trainer(
    module_file=TRAINER_MODULE_FILE,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    hyperparameters=tuner.outputs["best_hyperparameters"],
    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=20),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=10),
)
context.run(trainer)

running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying trainer.py -> build/lib
copying tuner.py -> build/lib
copying transform.py -> build/lib
installing to /tmp/tmpi4e8p37q
running install
running install_lib
copying build/lib/tuner.py -> /tmp/tmpi4e8p37q
copying build/lib/transform.py -> /tmp/tmpi4e8p37q
copying build/lib/trainer.py -> /tmp/tmpi4e8p37q
running install_egg_info
running egg_info
creating tfx_user_code_Trainer.egg-info
writing tfx_user_code_Trainer.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Trainer.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Trainer.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
Copying tfx_user_code_Trainer.egg-info to /tmp/tmpi4e8p37q/tfx_user_code_Trainer-0.0+e9cb44594d6869e74ad0813cd4ed047f43032

/workspaces/ml-pipeline/venv/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


Processing /workspaces/ml-pipeline/main-pipeline/_wheels/tfx_user_code_Trainer-0.0+e9cb44594d6869e74ad0813cd4ed047f43032fa52601de7e6604f061eaecdca4-py3-none-any.whl



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Epoch 1/10
20/20 [==============================] - 1s 17ms/step - loss: 0.4485 - accuracy: 0.8219 - auc: 0.8904 - precision: 0.8487 - recall: 0.8195 - val_loss: 0.4433 - val_accuracy: 0.8406 - val_auc: 0.8989 - val_precision: 0.8478 - val_recall: 0.8715
Epoch 2/10
20/20 [==============================] - 0s 4ms/step - loss: 0.3476 - accuracy: 0.8500 - auc: 0.9277 - precision: 0.8672 - recall: 0.8624 - val_loss: 0.4531 - val_accuracy: 0.8281 - val_auc: 0.9147 - val_precision: 0.8018 - val_recall: 0.9355
Epoch 3/10
20/20 [==============================] - 0s 4ms/step - loss: 0.3554 - accuracy: 0.8422 - auc: 0.9214 - precision: 0.8444 - recall: 0.8711 - val_loss: 0.4230 - val_accuracy: 0.8562 - val_auc: 0.9252 - val_precision: 0.8477 - val_recall: 0.9126
Epoch 4/10
20/20 [==============================] - 0s 5ms/step - loss: 0.3058 - accuracy: 0.8875 - auc: 0.9408 - precision: 0.8955 - recall: 0.9006 - val_loss: 0.4414 - val_accuracy: 0.8188 - val_auc: 0.9011 - val_precision: 0.7885 - va

...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: ../main-pipeline/Trainer/model/7/Format-Serving/assets


INFO:tensorflow:Assets written to: ../main-pipeline/Trainer/model/7/Format-Serving/assets


ExecutionResult(
    component_id: Trainer
    execution_id: 7
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Komponen **`Trainer`** melatih model klasifikasi Deep Neural Network menggunakan konfigurasi hyperparameter terbaik dari komponen `Tuner`. Proses pelatihan ini mencakup:

- **Eksekusi Modul Pelatihan:** Menjalankan skrip `modules/trainer.py` yang mengombinasikan transform graph dan arsitektur model.
- **Penerapan Best Hyperparameters:** Menggunakan keluaran `best_hyperparameters` untuk mengonfigurasi arsitektur dan proses pelatihan secara optimal.
- **Ekspor Model:** Menghasilkan artefak SavedModel yang siap pakai, lengkap dengan grafik prapemrosesan dan signature `serving_default` untuk deployment via TensorFlow Serving.

## **Model Resolver & Evaluator**

In [8]:
# Mendapatkan model terbaik sebelumnya untuk dijadikan baseline
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing),
).with_id("latest_blessed_model_resolver")

context.run(model_resolver)

# Menguji performa model menggunakan nama kolom label mentah
eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key="HeartDisease")],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name="BinaryAccuracy"),
                tfma.MetricConfig(class_name="ExampleCount"),
                tfma.MetricConfig(class_name="AUC"),
                tfma.MetricConfig(
                    class_name="BinaryAccuracy",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.6}
                        ),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={"value": -1e-10},
                        ),
                    ),
                ),
            ]
        )
    ],
)

evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    baseline_model=model_resolver.outputs["model"],
    eval_config=eval_config,
)
context.run(evaluator)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 9
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Sebelum model baru dipublikasikan ke lingkungan produksi, kelayakannya diuji secara ketat melalui dua komponen utama:

- **`Resolver`:** Mencari dan memuat versi model sebelumnya yang berstatus blessed (layak rilis) untuk dijadikan sebagai model pembanding (baseline).
- **`Evaluator`:** Menganalisis performa model menggunakan pustaka TensorFlow Model Analysis (TFMA) berdasarkan metrik seperti `BinaryAccuracy`, `AUC`, dan `ExampleCount`.
- **Kriteria Validasi:** Model kandidat baru hanya akan dinyatakan blessed jika memenuhi dua syarat:
  - **Nilai Minimal:** Memiliki akurasi (`BinaryAccuracy`) minimal **60%** (0.6).
  - **Komparasi Baseline:** Performanya tidak boleh mengalami penurunan dibandingkan model baseline sebelumnya.

## **Model Deployment Artifact**

In [9]:
SERVING_MODEL_DIR = os.path.join("..", "serving_model", "heart-failure-model")

pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    ),
)
context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 10
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Komponen **`Pusher`** mengotomatiskan proses penempatan model ke lingkungan produksi berdasarkan hasil evaluasi akhir. Alur kerjanya meliputi:

- **Verifikasi Status Blessing:** Memeriksa status kelayakan model dari komponen `Evaluator` (`model_blessing`).
- **Penyalinan Artefak Model:** Jika model dinyatakan blessed, `Pusher` akan menyalin berkas SavedModel dari direktori kerja pipeline ke direktori tujuan (`serving_model/heart-failure-model`).
- **Kesiapan Serving:** Menyiapkan struktur direktori model agar dapat langsung di-deploy menggunakan TensorFlow Serving (misalnya via container Docker).